# Training YOLOv8 Smart Gate

Notebook ini memakai `dataset_merged/` sebagai dataset utama. `plat-nomor/` digunakan sebagai fallback jika dataset gabungan tidak tersedia.

In [ ]:
!pip install -q ultralytics pyyaml gdown

from pathlib import Path
import zipfile
import gdown
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU CUDA tidak aktif. Di Colab pilih Runtime > Change runtime type > T4 GPU lalu jalankan ulang.')
print('GPU:', torch.cuda.get_device_name(0))

# Dataset gabungan diunduh langsung dari Google Drive.
FILE_ID = '138RROLwz9wkpmgJiwwYOl3Tf1UahDKZz'
ZIP_PATH = Path('/content/smartgate-all-datasets.zip')
if not ZIP_PATH.is_file():
    gdown.download(id=FILE_ID, output=str(ZIP_PATH), quiet=False)
if not ZIP_PATH.is_file() or ZIP_PATH.stat().st_size < 1000000:
    raise RuntimeError(f'Download ZIP gagal atau file terlalu kecil: {ZIP_PATH}')
with zipfile.ZipFile(ZIP_PATH) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f'ZIP rusak pada entry: {bad_member}')
    archive.extractall('/content')

candidates = [path.parent for path in Path('/content').rglob('data.yaml') if path.parent.name in ('dataset_merged', 'plat-nomor')]
preferred = [path for path in candidates if path.name == 'dataset_merged']
DATA_ROOT = next(iter(preferred or candidates), None)
if DATA_ROOT is None:
    raise FileNotFoundError('data.yaml tidak ditemukan setelah download dan ekstraksi ZIP.')
DATA_YAML = DATA_ROOT / 'data.yaml'
print('ZIP:', ZIP_PATH, f'({ZIP_PATH.stat().st_size / 1024**3:.2f} GB)')
print('Dataset:', DATA_ROOT)
print('Config sumber:', DATA_YAML)

In [ ]:
import yaml

config = yaml.safe_load(DATA_YAML.read_text())
config['path'] = str(DATA_ROOT)
config['train'] = 'train/images'
config['val'] = 'valid/images'
config['test'] = 'test/images'
config['nc'] = 1
config['names'] = ['number-plate']
RUNTIME_YAML = Path('/content/dataset_runtime.yaml')
RUNTIME_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(RUNTIME_YAML.read_text())

for split in ('train', 'val', 'test'):
    image_dir = DATA_ROOT / config[split]
    label_dir = DATA_ROOT / config[split].replace('/images', '/labels')
    if not image_dir.is_dir():
        raise FileNotFoundError(f'{split} images tidak ditemukan: {image_dir}')
    if not label_dir.is_dir():
        raise FileNotFoundError(f'{split} labels tidak ditemukan: {label_dir}')
    images = {p.stem for p in image_dir.iterdir() if p.is_file()}
    labels = {p.stem for p in label_dir.glob('*.txt')}
    missing_labels = sorted(images - labels)
    orphan_labels = sorted(labels - images)
    if missing_labels or orphan_labels:
        raise ValueError(f'{split}: missing labels={len(missing_labels)}, orphan labels={len(orphan_labels)}')
    for label_path in label_dir.glob('*.txt'):
        for line_no, line in enumerate(label_path.read_text().splitlines(), 1):
            fields = line.split()
            if len(fields) != 5 or int(fields[0]) != 0 or not all(0.0 <= float(v) <= 1.0 for v in fields[1:]):
                raise ValueError(f'Anotasi invalid: {label_path}:{line_no} -> {line}')
    print(f'{split}: {len(images)} images, {len(labels)} labels, valid')
print('Struktur dan anotasi dataset valid.')

In [ ]:
def count_labels(folder):
    files = list(Path(folder).glob('*.txt'))
    objects = sum(len(path.read_text().splitlines()) for path in files)
    return len(files), objects

for split in ('train', 'valid', 'test'):
    label_dir = DATA_ROOT / split / 'labels'
    files, objects = count_labels(label_dir)
    print(f'{split}: {files} file label, {objects} anotasi')

## Training

Gunakan GPU Colab: **Runtime > Change runtime type > T4 GPU**. Kotak anotasi harus tepat mengelilingi papan plat, bukan bumper, kamera, gril, atau seluruh kendaraan.

In [ ]:
from ultralytics import YOLO
import torch

device = 0
batch = 16
print('Device:', device)

RUN_DIR = DATA_ROOT.parent / 'runs' / 'detect'
model = YOLO('yolov8n.pt')
model.train(
    data=str(RUNTIME_YAML),
    epochs=50,
    imgsz=640,
    batch=batch,
    device=device,
    project=str(RUN_DIR),
    name='smartgate-colab',
    patience=12,
    hsv_h=0.015,
    hsv_s=0.55,
    hsv_v=0.35,
    degrees=5.0,
    translate=0.08,
    scale=0.35,
    fliplr=0.0,
    mosaic=0.7,
    mixup=0.05,
    close_mosaic=10,
    pretrained=True,
    plots=True,
    val=True,
    seed=42,
    workers=2,
    cache=False,
    exist_ok=True,
)
print('Training selesai.')

In [ ]:
best_pt = RUN_DIR / 'smartgate-colab' / 'weights' / 'best.pt'
if not best_pt.is_file():
    raise FileNotFoundError(best_pt)
trained = YOLO(str(best_pt))
val_metrics = trained.val(data=str(RUNTIME_YAML), split='val', imgsz=640, device=device)
test_metrics = trained.val(data=str(RUNTIME_YAML), split='test', imgsz=640, device=device)
print('best.pt:', best_pt)
print(f'Validation mAP50: {val_metrics.box.map50:.4f}')
print(f'Validation mAP50-95: {val_metrics.box.map:.4f}')
print(f'Test mAP50: {test_metrics.box.map50:.4f}')
print(f'Test mAP50-95: {test_metrics.box.map:.4f}')

In [ ]:
from google.colab import files
print('Bobot selesai:', best_pt)
print('Download best.pt untuk deployment lokal:')
files.download(str(best_pt))

Setelah training selesai, salin `best.pt` ke:

`Tugas Citra Digital/prototype/runs/detect/smartgate-colab/weights/best.pt`

Lalu jalankan aplikasi dengan `SGS_YOLO_WEIGHTS` menunjuk ke file tersebut dan restart Flask.